# January 2025 Data Cleaning

This notebook loads raw PJM, NYISO, and NOAA data, cleans and validates the
hourly observations, merges electricity and weather data using UTC timestamps,
and exports standardized processed files.

`timestamp_utc` is the canonical key for joins, ordering, and later modeling.
`timestamp_local` is retained for local calendar features and reporting.


In [1]:
import pandas as pd

from electricity_forecasting.config import (
    MARKETS,
    PROCESSED_DATA_DIR,
)


## 1. Verify configuration and input files


In [2]:
print("PJM location:", MARKETS["pjm"]["location"])
print("NYISO location:", MARKETS["nyiso"]["location"])

for market_name, market_config in MARKETS.items():
    print(f"\n{market_name.upper()}")

    for key in ["price_file", "load_file", "weather_file"]:
        file_path = market_config[key]
        print(f"{key}: {file_path.exists()} — {file_path}")
        assert file_path.exists(), f"Missing input file: {file_path}"


PJM location: PSEG
NYISO location: HUD VL

PJM
price_file: True — C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\raw\pjm\da_hrl_lmps_PJM_PS.csv
load_file: True — C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\raw\pjm\hrl_load_metered_PJM_PS.csv
weather_file: True — C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\raw\noaa\WeatherData Jan25 Newark.csv

NYISO
price_file: True — C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\raw\nyiso\nyiso_hudson_valley_jan2025_LMP_DATA.csv
load_file: True — C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\raw\nyiso\nyiso_hudson_valley_jan2025palIntegrated_HV_loaddata.csv
weather_file: True — C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\raw\noaa\WeatherData Jan25 Stewart.csv


## 2. Load raw data


In [3]:
pjm_config = MARKETS["pjm"]
nyiso_config = MARKETS["nyiso"]

pjm_price_raw = pd.read_csv(pjm_config["price_file"])
pjm_load_raw = pd.read_csv(pjm_config["load_file"])
pjm_weather_raw = pd.read_csv(
    pjm_config["weather_file"],
    low_memory=False,
)

nyiso_price_raw = pd.read_csv(nyiso_config["price_file"])
nyiso_load_raw = pd.read_csv(nyiso_config["load_file"])
nyiso_weather_raw = pd.read_csv(
    nyiso_config["weather_file"],
    low_memory=False,
)

raw_datasets = {
    "PJM price": pjm_price_raw,
    "PJM load": pjm_load_raw,
    "PJM weather": pjm_weather_raw,
    "NYISO price": nyiso_price_raw,
    "NYISO load": nyiso_load_raw,
    "NYISO weather": nyiso_weather_raw,
}

for name, data in raw_datasets.items():
    print(
        f"{name:15} "
        f"rows={data.shape[0]:6} "
        f"columns={data.shape[1]:3}"
    )

assert pjm_price_raw.shape == (744, 14)
assert pjm_load_raw.shape == (744, 8)
assert nyiso_price_raw.shape == (744, 7)
assert nyiso_load_raw.shape == (744, 6)


PJM price       rows=   744 columns= 14
PJM load        rows=   744 columns=  8
PJM weather     rows= 12970 columns=125
NYISO price     rows=   744 columns=  7
NYISO load      rows=   744 columns=  6
NYISO weather   rows=  9081 columns=125


## 3. Clean PJM electricity data


In [4]:
pjm_price = pjm_price_raw.copy()

pjm_price["timestamp_local"] = (
    pd.to_datetime(
        pjm_price["datetime_beginning_ept"],
        format="%m/%d/%Y %I:%M:%S %p",
    )
    .dt.tz_localize("America/New_York")
)

pjm_price["timestamp_utc"] = pd.to_datetime(
    pjm_price["datetime_beginning_utc"],
    format="%m/%d/%Y %I:%M:%S %p",
    utc=True,
)

pjm_price = pjm_price[
    [
        "timestamp_local",
        "timestamp_utc",
        "pnode_id",
        "pnode_name",
        "total_lmp_da",
        "system_energy_price_da",
        "congestion_price_da",
        "marginal_loss_price_da",
    ]
].rename(
    columns={
        "pnode_id": "location_id",
        "pnode_name": "location",
        "total_lmp_da": "day_ahead_price_usd_mwh",
        "system_energy_price_da": "energy_component_usd_mwh",
        "congestion_price_da": "congestion_component_usd_mwh",
        "marginal_loss_price_da": "loss_component_usd_mwh",
    }
)

pjm_price = (
    pjm_price
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)


In [5]:
pjm_load = pjm_load_raw.copy()

pjm_load["timestamp_local"] = (
    pd.to_datetime(
        pjm_load["datetime_beginning_ept"],
        format="%m/%d/%Y %I:%M:%S %p",
    )
    .dt.tz_localize("America/New_York")
)

pjm_load["timestamp_utc"] = pd.to_datetime(
    pjm_load["datetime_beginning_utc"],
    format="%m/%d/%Y %I:%M:%S %p",
    utc=True,
)

pjm_load = pjm_load[
    [
        "timestamp_local",
        "timestamp_utc",
        "zone",
        "load_area",
        "mw",
        "is_verified",
    ]
].rename(
    columns={
        "mw": "actual_load_mw",
    }
)

pjm_load = (
    pjm_load
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)


## 4. Clean NYISO electricity data


In [16]:
nyiso_price = nyiso_price_raw.copy()

nyiso_price["timestamp_local"] = (
    pd.to_datetime(
        nyiso_price["Time Stamp"],
        format="%Y-%m-%d %H:%M:%S",
    )
    .dt.tz_localize("America/New_York")
)

nyiso_price["timestamp_utc"] = (
    nyiso_price["timestamp_local"]
    .dt.tz_convert("UTC")
)

nyiso_price = nyiso_price[
    [
        "timestamp_local",
        "timestamp_utc",
        "PTID",
        "Name",
        "LBMP ($/MWHr)",
        "Marginal Cost Losses ($/MWHr)",
        "Marginal Cost Congestion ($/MWHr)",
    ]
].rename(
    columns={
        "PTID": "location_id",
        "Name": "location",
        "LBMP ($/MWHr)": "day_ahead_price_usd_mwh",
        "Marginal Cost Losses ($/MWHr)": "loss_component_usd_mwh",
        "Marginal Cost Congestion ($/MWHr)": "congestion_component_usd_mwh",
    }
)

nyiso_price["energy_component_usd_mwh"] = (
    nyiso_price["day_ahead_price_usd_mwh"]
    - nyiso_price["loss_component_usd_mwh"]
    - nyiso_price["congestion_component_usd_mwh"]
)

nyiso_price = (
    nyiso_price
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)


In [7]:
nyiso_load = nyiso_load_raw.copy()

nyiso_load["timestamp_local"] = (
    pd.to_datetime(
        nyiso_load["Time Stamp"],
        format="%Y-%m-%d %H:%M:%S",
    )
    .dt.tz_localize("America/New_York")
)

nyiso_load["timestamp_utc"] = (
    nyiso_load["timestamp_local"]
    .dt.tz_convert("UTC")
)

nyiso_load = nyiso_load[
    [
        "timestamp_local",
        "timestamp_utc",
        "Time Zone",
        "Name",
        "PTID",
        "Integrated Load",
    ]
].rename(
    columns={
        "Time Zone": "source_time_zone",
        "Name": "load_location",
        "PTID": "load_location_id",
        "Integrated Load": "actual_load_mw",
    }
)

nyiso_load = (
    nyiso_load
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)


## 5. Validate cleaned electricity tables


In [8]:
def validate_hourly_table(
    df: pd.DataFrame,
    table_name: str,
    value_column: str,
) -> None:
    expected_rows = 744

    assert len(df) == expected_rows, (
        f"{table_name}: expected {expected_rows} rows, "
        f"but found {len(df)}"
    )

    assert df["timestamp_local"].notna().all(), (
        f"{table_name}: missing local timestamps"
    )

    assert df["timestamp_utc"].notna().all(), (
        f"{table_name}: missing UTC timestamps"
    )

    assert df["timestamp_local"].is_unique, (
        f"{table_name}: duplicate local timestamps"
    )

    assert df["timestamp_utc"].is_unique, (
        f"{table_name}: duplicate UTC timestamps"
    )

    assert df["timestamp_utc"].is_monotonic_increasing, (
        f"{table_name}: UTC timestamps are not sorted"
    )

    assert df[value_column].notna().all(), (
        f"{table_name}: missing values in {value_column}"
    )

    print(f"{table_name} passed validation.")


validate_hourly_table(
    pjm_price,
    "PJM price",
    "day_ahead_price_usd_mwh",
)

validate_hourly_table(
    pjm_load,
    "PJM load",
    "actual_load_mw",
)

validate_hourly_table(
    nyiso_price,
    "NYISO price",
    "day_ahead_price_usd_mwh",
)

validate_hourly_table(
    nyiso_load,
    "NYISO load",
    "actual_load_mw",
)

assert pjm_load["is_verified"].all(), (
    "PJM load contains unverified observations"
)


PJM price passed validation.
PJM load passed validation.
NYISO price passed validation.
NYISO load passed validation.


## 6. Clean NOAA weather data


In [9]:
JANUARY_HOURS = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-01-31 23:00:00",
    freq="h",
)


def clean_weather(
    raw_weather: pd.DataFrame,
    station_code: str,
) -> pd.DataFrame:
    weather = raw_weather.copy()

    weather["observed_at"] = pd.to_datetime(
        weather["DATE"],
        errors="coerce",
    )

    weather = weather[
        weather["observed_at"].between(
            "2025-01-01 00:00:00",
            "2025-01-31 23:59:59",
        )
        & weather["REPORT_TYPE"].isin(
            ["FM-12", "FM-15", "FM-16"]
        )
    ].copy()

    source_columns = {
        "HourlyDryBulbTemperature": "temperature_c",
        "HourlyDewPointTemperature": "dew_point_c",
        "HourlyRelativeHumidity": "relative_humidity_pct",
        "HourlyWindSpeed": "wind_speed_mps",
    }

    quality_flag_columns = []

    for source_column, clean_column in source_columns.items():
        raw_values = weather[source_column].astype("string")
        flag_column = f"{clean_column}_quality_flagged"

        weather[flag_column] = (
            raw_values.notna()
            & raw_values.str.contains(
                r"[A-Za-z]",
                regex=True,
                na=False,
            )
        )

        weather[clean_column] = pd.to_numeric(
            raw_values.str.extract(
                r"([-+]?\d*\.?\d+)",
                expand=False,
            ),
            errors="coerce",
        )

        quality_flag_columns.append(flag_column)

    value_columns = [
        "temperature_c",
        "dew_point_c",
        "relative_humidity_pct",
        "wind_speed_mps",
    ]

    invalid_temperature = ~weather["temperature_c"].between(
        -50,
        50,
    )

    invalid_dew_point = ~weather["dew_point_c"].between(
        -60,
        40,
    )

    invalid_humidity = ~weather[
        "relative_humidity_pct"
    ].between(
        0,
        100,
    )

    invalid_wind_speed = ~weather[
        "wind_speed_mps"
    ].between(
        0,
        75,
    )

    weather["weather_value_rejected"] = (
        invalid_temperature
        | invalid_dew_point
        | invalid_humidity
        | invalid_wind_speed
    )

    weather.loc[
        weather["weather_value_rejected"],
        value_columns,
    ] = pd.NA

    weather["weather_quality_flagged"] = weather[
        quality_flag_columns
    ].any(axis=1)

    weather["timestamp_local"] = (
        weather["observed_at"].dt.floor("h")
    )

    report_priority = {
        "FM-15": 1,
        "FM-12": 2,
        "FM-16": 3,
    }

    weather["report_priority"] = weather[
        "REPORT_TYPE"
    ].map(report_priority)

    hourly = (
        weather.sort_values(
            [
                "timestamp_local",
                "report_priority",
                "observed_at",
            ]
        )
        .drop_duplicates(
            subset="timestamp_local",
            keep="first",
        )
        .set_index("timestamp_local")
        .reindex(JANUARY_HOURS)
    )

    hourly["weather_missing"] = hourly[
        value_columns
    ].isna().any(axis=1)

    hourly["weather_imputed"] = False
    hourly["weather_station"] = station_code
    hourly.index.name = "timestamp_local"

    hourly = hourly.reset_index()

    hourly["timestamp_local"] = (
        hourly["timestamp_local"]
        .dt.tz_localize("America/New_York")
    )

    hourly["timestamp_utc"] = (
        hourly["timestamp_local"]
        .dt.tz_convert("UTC")
    )

    output_columns = [
        "timestamp_local",
        "timestamp_utc",
        "observed_at",
        "REPORT_TYPE",
        "temperature_c",
        "dew_point_c",
        "relative_humidity_pct",
        "wind_speed_mps",
        "weather_quality_flagged",
        "weather_value_rejected",
        "weather_missing",
        "weather_imputed",
        "weather_station",
    ]

    return hourly[output_columns]


In [10]:
newark_weather = clean_weather(
    pjm_weather_raw,
    station_code="USW00014734",
)

stewart_weather = clean_weather(
    nyiso_weather_raw,
    station_code="USW00014714",
)

weather_value_columns = [
    "temperature_c",
    "dew_point_c",
    "relative_humidity_pct",
    "wind_speed_mps",
]


def summarize_weather(
    weather: pd.DataFrame,
    station_name: str,
) -> None:
    print(f"\n{station_name}")
    print("Rows:", len(weather))
    print(
        "Duplicate UTC hours:",
        weather["timestamp_utc"].duplicated().sum(),
    )
    print(
        "Missing hours or values:",
        weather["weather_missing"].sum(),
    )
    print(
        "Quality-flagged observations:",
        weather["weather_quality_flagged"]
        .fillna(False)
        .sum(),
    )
    print(
        "Rejected observations:",
        weather["weather_value_rejected"]
        .fillna(False)
        .sum(),
    )
    print("\nMissing values by column:")
    print(weather[weather_value_columns].isna().sum())

    assert len(weather) == 744
    assert weather["timestamp_utc"].notna().all()
    assert weather["timestamp_utc"].is_unique
    assert weather["timestamp_utc"].is_monotonic_increasing


summarize_weather(newark_weather, "Newark")
summarize_weather(stewart_weather, "Stewart")



Newark
Rows: 744
Duplicate UTC hours: 0
Missing hours or values: 3
Quality-flagged observations: 0
Rejected observations: 0

Missing values by column:
temperature_c            3
dew_point_c              3
relative_humidity_pct    3
wind_speed_mps           3
dtype: int64

Stewart
Rows: 744
Duplicate UTC hours: 0
Missing hours or values: 7
Quality-flagged observations: 7
Rejected observations: 4

Missing values by column:
temperature_c            7
dew_point_c              7
relative_humidity_pct    7
wind_speed_mps           7
dtype: int64


### NOAA weather units

The NOAA Local Climatological Data CSV files use metric/SI units.

The selected variables are interpreted as:

- `HourlyDryBulbTemperature`: degrees Celsius
- `HourlyDewPointTemperature`: degrees Celsius
- `HourlyRelativeHumidity`: percent
- `HourlyWindSpeed`: meters per second

No unit conversion is applied. Values outside documented plausible ranges
are rejected and recorded through the `weather_value_rejected` indicator.

## 7. Merge price, load, and weather data


In [11]:
pjm_electricity = pjm_price.merge(
    pjm_load.drop(
        columns=["timestamp_local"],
        errors="ignore",
    ),
    on="timestamp_utc",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

print(pjm_electricity["_merge"].value_counts())
assert len(pjm_electricity) == 744
assert pjm_electricity["_merge"].eq("both").all()

pjm_electricity = (
    pjm_electricity
    .drop(columns="_merge")
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)

pjm_processed = pjm_electricity.merge(
    newark_weather.drop(
        columns=["timestamp_local"],
        errors="ignore",
    ),
    on="timestamp_utc",
    how="left",
    validate="one_to_one",
)

pjm_processed["market"] = "PJM"
pjm_processed["pricing_location"] = "PSEG"
pjm_processed["load_data_type"] = "actual_metered"

print("PJM processed shape:", pjm_processed.shape)


_merge
both          744
left_only       0
right_only      0
Name: count, dtype: int64
PJM processed shape: (744, 26)


In [12]:
nyiso_electricity = nyiso_price.merge(
    nyiso_load.drop(
        columns=["timestamp_local"],
        errors="ignore",
    ),
    on="timestamp_utc",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

print(nyiso_electricity["_merge"].value_counts())
assert len(nyiso_electricity) == 744
assert nyiso_electricity["_merge"].eq("both").all()

nyiso_electricity = (
    nyiso_electricity
    .drop(columns="_merge")
    .sort_values("timestamp_utc")
    .reset_index(drop=True)
)

nyiso_processed = nyiso_electricity.merge(
    stewart_weather.drop(
        columns=["timestamp_local"],
        errors="ignore",
    ),
    on="timestamp_utc",
    how="left",
    validate="one_to_one",
)

nyiso_processed["market"] = "NYISO"
nyiso_processed["pricing_location"] = "HUD VL"
nyiso_processed["load_data_type"] = "actual_integrated"

print("NYISO processed shape:", nyiso_processed.shape)


_merge
both          744
left_only       0
right_only      0
Name: count, dtype: int64
NYISO processed shape: (744, 26)


## 8. Final validation


In [13]:
for name, data in {
    "PJM processed": pjm_processed,
    "NYISO processed": nyiso_processed,
}.items():
    print(f"\n{name}")
    print("Rows:", len(data))
    print(
        "Duplicate UTC timestamps:",
        data["timestamp_utc"].duplicated().sum(),
    )
    print(
        "Missing prices:",
        data["day_ahead_price_usd_mwh"].isna().sum(),
    )
    print(
        "Missing load:",
        data["actual_load_mw"].isna().sum(),
    )
    print(
        "Rows with missing weather:",
        data["weather_missing"].fillna(True).sum(),
    )

    assert len(data) == 744
    assert data["timestamp_utc"].is_unique
    assert data["timestamp_utc"].is_monotonic_increasing
    assert data["day_ahead_price_usd_mwh"].notna().all()
    assert data["actual_load_mw"].notna().all()



PJM processed
Rows: 744
Duplicate UTC timestamps: 0
Missing prices: 0
Missing load: 0
Rows with missing weather: 3

NYISO processed
Rows: 744
Duplicate UTC timestamps: 0
Missing prices: 0
Missing load: 0
Rows with missing weather: 7


## 9. Export processed datasets


In [14]:
PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

pjm_output_path = (
    PROCESSED_DATA_DIR
    / "pjm_pseg_january_2025_electricity.csv"
)

nyiso_output_path = (
    PROCESSED_DATA_DIR
    / "nyiso_hudson_valley_january_2025_electricity.csv"
)

pjm_processed.to_csv(
    pjm_output_path,
    index=False,
)

nyiso_processed.to_csv(
    nyiso_output_path,
    index=False,
)

print("Saved:", pjm_output_path.resolve())
print("Saved:", nyiso_output_path.resolve())


Saved: C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\processed\pjm_pseg_january_2025_electricity.csv
Saved: C:\Users\david\Desktop\Data Science\DATA 698\electricity-price-forecasting\data\processed\nyiso_hudson_valley_january_2025_electricity.csv


In [15]:
assert pjm_output_path.exists()
assert nyiso_output_path.exists()

pjm_exported = pd.read_csv(pjm_output_path)
nyiso_exported = pd.read_csv(nyiso_output_path)

print("PJM exported rows:", len(pjm_exported))
print("NYISO exported rows:", len(nyiso_exported))

assert len(pjm_exported) == 744
assert len(nyiso_exported) == 744


PJM exported rows: 744
NYISO exported rows: 744
